# Pipeline de Procesamiento de Datos de AppEEARS

Este *notebook* implementa un flujo completo de transformación y carga (TL) para obtener datos históricos sobre diferentes aspectos a tener en cuenta sobre una ciudad o país, como pueden ser su índice de vegetación y de edificación/asfalto. Los datos obtenidos de AppEEARS se cargan en este *notebook*, para ser preprocesados y limpiados.

In [1]:
# NumPy: Es la biblioteca fundamental para la computación científica en Python.
# Proporciona estructuras de datos de tipo 'array' que son mucho más eficientes 
# que las listas nativas. Se usa para operaciones matemáticas complejas y 
# transformaciones de matrices de datos.
import numpy as np           

# Pandas: Es la herramienta estrella para el análisis y manipulación de datos. 
# Introduce los 'DataFrames', que permiten trabajar con tablas (filas y columnas) 
# de forma intuitiva, realizar filtros, uniones (merges) y limpiezas de valores 
# nulos o erróneos en los registros de contaminación.
import pandas as pd          

# OS (Operating System): Este módulo permite que Python interactúe con el 
# sistema operativo de tu ordenador. Es vital para la gestión de rutas (paths), 
# permitiendo que el script localice, lea y guarde archivos en carpetas 
# específicas sin importar si el usuario trabaja en Windows o Linux.
import os

from pathlib import Path

In [4]:
# Definimos la ruta base de datasets.
# La carpeta "datasets" está un nivel por encima de la carpeta actual del proyecto.
BASE_PATH = Path("..", "..") / "datasets"

# Carpeta donde están los archivos descargados de AppEEARS
APPEEARS_DIR = BASE_PATH / "AppEEARS"

# Rutas de los archivos de vegetación y edificación
ruta_vegetacion = APPEEARS_DIR / "TFM-Urban-Indices-Spain-MOD13Q1-061-results.csv"
ruta_edificacion = APPEEARS_DIR / "TFM-Urban-Indices-Spain-MOD09A1-061-results.csv"

In [5]:
# Dataframe de cada índice
df_vegetacion = pd.read_csv(ruta_vegetacion, encoding="latin1", skip_blank_lines=True)
df_edificacion = pd.read_csv(ruta_edificacion, encoding="latin1", skip_blank_lines=True)

### Índice de vegetación

De todas las variables que se nos dan en el el CSV correspondiente al índice de vegetación, solo nos interesan unas pocas.

In [6]:
# =============================================================================
# ANÁLISIS DE PRIMER DATAFRAME: VEGETACIÓN
# =============================================================================

# Observamos todas las columnas del dataframe de vegetación para entender su estructura y contenido.
df_vegetacion.info()

<class 'pandas.DataFrame'>
RangeIndex: 3324 entries, 0 to 3323
Data columns (total 29 columns):
 #   Column                                                                       Non-Null Count  Dtype  
---  ------                                                                       --------------  -----  
 0   Category                                                                     3324 non-null   str    
 1   ID                                                                           3324 non-null   str    
 2   Latitude                                                                     3324 non-null   float64
 3   Longitude                                                                    3324 non-null   float64
 4   Date                                                                         3324 non-null   str    
 5   MODIS_Tile                                                                   3324 non-null   str    
 6   MOD13Q1_061_Line_Y_250m                            

En concreto, nos interesan las siguientes variables:

* **ID**: el nombre de la capital
* **Date**: la fecha de la observación.
* **MOD13Q1_061__250m_16_days_NDVI**: esta variable mide el índice de vegetación. Viene multiplicada por 10000.
* **MOD13Q1_061__250m_16_days_VI_Quality_MODLAND_Description**: nos dice en texto si el dato es de buena calidad o si había nubes.
* **MOD13Q1_061__250m_16_days_VI_Quality_VI_Usefulness_Description**: clasifica los datos en una escala de 16 niveles, desde "Perfecto" hasta "Inútil".

In [7]:
# Seleccionamos las columnas de interés: Identificadores, Valores y Metadatos de Calidad
columnas_interes = [
    'ID', 'Date', 
    'MOD13Q1_061__250m_16_days_NDVI', 
    'MOD13Q1_061__250m_16_days_VI_Quality_MODLAND_Description',
    'MOD13Q1_061__250m_16_days_VI_Quality_VI_Usefulness_Description'
]

df_vegetacion = df_vegetacion[columnas_interes].copy()

# Renombramos para facilitar la lectura y evitar errores con nombres largos de la NASA
df_vegetacion.columns = ['Ciudad', 'Fecha', 'NDVI', 'Calidad_Texto', 'Utilidad_Texto']

# Convertimos la fecha a formato datetime de Python para poder hacer cálculos temporales
df_vegetacion['Fecha'] = pd.to_datetime(df_vegetacion['Fecha'])

# El NDVI de MODIS viene escalado (multiplicado por 10.000). Lo pasamos a su rango real (0 a 1)
df_vegetacion['NDVI'] = df_vegetacion['NDVI'] * 0.0001

In [8]:
# =============================================================================
# CLASIFICACIÓN Y FILTRADO
# =============================================================================

# Definimos nuestra "Máscara de Aceptación" (Filtro Híbrido)
# Aceptamos datos si:
# - La calidad MODLAND es 'good' o permite revisión ('check')
# - Y la utilidad NO es 'Lowest quality' ni 'Not useful'
mask_aceptables = (
    df_vegetacion['Calidad_Texto'].str.contains('good|check', case=False, na=False) & 
    ~df_vegetacion['Utilidad_Texto'].str.contains('Lowest quality|Not useful', case=False, na=False)
)

# Creamos una nueva columna 'NDVI_Procesado'
# - Si el dato es aceptable, mantenemos el valor real.
# - Si el dato es "basura" (nubes, sombra, fallo), lo ponemos como NaN (nulo).
df_vegetacion['NDVI_Procesado'] = df_vegetacion['NDVI']
df_vegetacion.loc[~mask_aceptables, 'NDVI_Procesado'] = np.nan

In [9]:
# =============================================================================
# INTERPOLACIÓN (RELLENADO DE HUECOS)
# =============================================================================

# Importante: Ordenar por Ciudad y Fecha para que la interpolación no mezcle datos
df_vegetacion = df_vegetacion.sort_values(['Ciudad', 'Fecha'])

# Aplicamos la interpolación lineal por cada grupo de Ciudad.
# Esto traza una línea entre el último punto bueno y el siguiente para rellenar los nulos.
df_vegetacion['NDVI_Final'] = df_vegetacion.groupby('Ciudad')['NDVI_Procesado'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

In [10]:
# =============================================================================
# SUAVIZADO
# =============================================================================

# Como hemos rescatado datos de calidad intermedia, aplicamos una Media Móvil (Rolling Mean).
# Usamos una ventana de 3 periodos (aprox. 1.5 meses) para suavizar la curva fenológica.
df_vegetacion['NDVI_Suave'] = df_vegetacion.groupby('Ciudad')['NDVI_Final'].transform(
    lambda x: x.rolling(window=3, center=True, min_periods=1).mean()
)

In [11]:
# =============================================================================
#  DIAGNÓSTICO FINAL
# =============================================================================

total = len(df_vegetacion)
buenos = mask_aceptables.sum()
rellenados = total - buenos

print("-" * 50)
print(f"RESUMEN DEL PROCESAMIENTO:")
print(f"1. Mediciones totales analizadas: {total}")
print(f"2. Mediciones reales mantenidas: {buenos} ({buenos/total:.1%})")
print(f"3. Mediciones interpoladas: {rellenados} ({rellenados/total:.1%})")
print("-" * 50)

df_vegetacion_final = df_vegetacion[['Ciudad', 'Fecha', 'NDVI_Final']].copy()
df_vegetacion_final.rename(columns={'NDVI_Final': 'NDVI'}, inplace=True)
df_vegetacion_final.head()

--------------------------------------------------
RESUMEN DEL PROCESAMIENTO:
1. Mediciones totales analizadas: 3324
2. Mediciones reales mantenidas: 3108 (93.5%)
3. Mediciones interpoladas: 216 (6.5%)
--------------------------------------------------


,Ciudad,Fecha,NDVI
0,A Corua,2012-12-18,0.000046
1,A Corua,2013-01-01,0.000036
2,A Corua,2013-01-17,0.000043
3,A Corua,2013-02-02,0.000043
4,A Corua,2013-02-18,0.000043


### Índice de edificación/asfalto

De todas las variables que se nos dan en el el CSV correspondiente al índice de edificación/asfalto, solo nos interesan unas pocas.

In [12]:
# =============================================================================
# ANÁLISIS DE PRIMER DATAFRAME: EDIFICACIÓN/ASFALTO 
# =============================================================================

df_edificacion.info()

<class 'pandas.DataFrame'>
RangeIndex: 6624 entries, 0 to 6623
Data columns (total 32 columns):
 #   Column                                                                       Non-Null Count  Dtype  
---  ------                                                                       --------------  -----  
 0   Category                                                                     6624 non-null   str    
 1   ID                                                                           6624 non-null   str    
 2   Latitude                                                                     6624 non-null   float64
 3   Longitude                                                                    6624 non-null   float64
 4   Date                                                                         6624 non-null   str    
 5   MODIS_Tile                                                                   6624 non-null   str    
 6   MOD09A1_061_Line_Y_500m                            

In [13]:
# Seleccionamos: Identificadores, Bandas para NDBI (b02 y b06) y Metadatos de calidad
columnas_ndbi = [
    'ID', 'Date', 
    'MOD09A1_061_sur_refl_b02', # Banda 2: NIR (Infrarrojo Cercano)
    'MOD09A1_061_sur_refl_b06', # Banda 6: SWIR (Infrarrojo de Onda Corta)
    'MOD09A1_061_sur_refl_qc_500m_MODLAND_Description',
    'MOD09A1_061_sur_refl_qc_500m_band_2_data_quality_four_bit_range_Description',
    'MOD09A1_061_sur_refl_qc_500m_band_6_data_quality_four_bit_range_Description'
]

df_edificacion = df_edificacion[columnas_ndbi].copy()

# Renombramos para trabajar con términos de teledetección
df_edificacion.columns = [
    'Ciudad', 'Fecha', 'NIR_raw', 'SWIR_raw', 
    'Calidad_General', 'Calidad_B2', 'Calidad_B6'
]

# Convertimos a formato fecha y escalamos la reflectancia (Factor de escala MODIS = 0.0001)
df_edificacion['Fecha'] = pd.to_datetime(df_edificacion['Fecha'])
df_edificacion['NIR'] = df_edificacion['NIR_raw'] * 0.0001
df_edificacion['SWIR'] = df_edificacion['SWIR_raw'] * 0.0001

In [14]:
# =============================================================================
# CÁLCULO DEL NDBI (Normalized Difference Built-up Index)
# =============================================================================
# Fórmula: (SWIR - NIR) / (SWIR + NIR)
# Interpretación: Valores > 0 suelen indicar suelo urbano/construido.
df_edificacion['NDBI_Calculado'] = (df_edificacion['SWIR'] - df_edificacion['NIR']) / (df_edificacion['SWIR'] + df_edificacion['NIR'])

In [15]:
# =============================================================================
# FILTRADO MULTICRITERIO (EL "PUNTO MEDIO" AVANZADO)
# =============================================================================

# Definimos los criterios de calidad:
# Aceptamos si la general es buena O si las bandas específicas son de alta calidad
mask_calidad = (
    df_edificacion['Calidad_General'].str.contains('ideal|produced', case=False, na=False) & 
    df_edificacion['Calidad_B2'].str.contains('highest|good', case=False, na=False) &
    df_edificacion['Calidad_B6'].str.contains('highest|good', case=False, na=False)
)

# Creamos la columna procesada: lo que no es de calidad se marca como nulo (NaN)
df_edificacion['NDBI_Limpio'] = df_edificacion['NDBI_Calculado']
df_edificacion.loc[~mask_calidad, 'NDBI_Limpio'] = np.nan

In [16]:
# =============================================================================
# INTERPOLACIÓN Y SUAVIZADO (TRATAMIENTO DE LA SERIE TEMPORAL)
# =============================================================================

# Ordenamos por ciudad y fecha para asegurar la coherencia temporal
df_edificacion = df_edificacion.sort_values(['Ciudad', 'Fecha'])

# Rellenamos los huecos (nubes/errores) mediante interpolación lineal por ciudad
df_edificacion['NDBI_Interpolado'] = df_edificacion.groupby('Ciudad')['NDBI_Limpio'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

# Aplicamos suavizado (Media Móvil de 3 periodos) para eliminar ruido residual
# Este es el valor que usarás para tus gráficas y modelos de regresión
df_edificacion['NDBI_Final'] = df_edificacion.groupby('Ciudad')['NDBI_Interpolado'].transform(
    lambda x: x.rolling(window=3, center=True, min_periods=1).mean()
)

In [17]:
# =============================================================================
# DIAGNÓSTICO DE CALIDAD
# =============================================================================

datos_totales = len(df_edificacion)
datos_buenos = mask_calidad.sum()
print("-" * 50)
print(f"ANÁLISIS DE EDIFICACIÓN (NDBI) COMPLETADO")
print(f"- Mediciones totales: {datos_totales}")
print(f"- Mediciones reales aceptadas: {datos_buenos} ({datos_buenos/datos_totales:.1%})")
print(f"- Mediciones interpoladas: {datos_totales - datos_buenos}")
print("-" * 50)

df_edificacion_final = df_edificacion[['Ciudad', 'Fecha', 'NDBI_Final']].copy()
df_edificacion_final.rename(columns={'NDBI_Final': 'NDBI'}, inplace=True)
df_edificacion_final.head()

--------------------------------------------------
ANÁLISIS DE EDIFICACIÓN (NDBI) COMPLETADO
- Mediciones totales: 6624
- Mediciones reales aceptadas: 6587 (99.4%)
- Mediciones interpoladas: 37
--------------------------------------------------


,Ciudad,Fecha,NDBI
0,A Corua,2012-12-26,-0.163860
1,A Corua,2013-01-01,-0.159821
2,A Corua,2013-01-09,-0.169538
3,A Corua,2013-01-17,-0.210487
4,A Corua,2013-01-25,-0.208741


Finalmente, nos descargamos ambos datasets.

In [18]:
# Creamos las rutas completas para los archivos de salida
ruta_salida_veg = APPEEARS_DIR / "índice_vegetación.csv"
ruta_salida_edif = APPEEARS_DIR / "índice_edificación.csv"

# Guardamos los archivos
df_vegetacion_final.to_csv(ruta_salida_veg, index=False, encoding="latin1")
df_edificacion_final.to_csv(ruta_salida_edif, index=False, encoding="latin1")